# Causal Conv1D Backward Operation

This notebook shows how to compute gradients for the causal depthwise 1D convolution using cuDNN.

Given the forward operation:

$$y = \text{Activation}(\text{CausalConv1D}(x, w) + b)$$

where Activation is Identity or SiLU, this notebook computes:

- $dx$ — gradient w.r.t. input $x$
- $dw$ — gradient w.r.t. weight $w$
- $db$ — gradient w.r.t. bias $b$

The `cudnn.ops.causal_conv1d` API integrates with `torch.autograd`, so backward is handled automatically via `.backward()`. We compare the cuDNN autograd result against a PyTorch reference.

## Prerequisites and Setup
This notebook requires an NVIDIA GPU (Hopper or later recommended).

**Environment setup** — make sure the cuDNN runtime library and the `cudnn` Python package are discoverable before launching the notebook:

- **Option A – pip install:**
  ```bash
  pip install nvidia-cudnn-frontend
  ```
- **Option B – set paths manually:**
  ```bash
  export LD_LIBRARY_PATH=/path/to/cudnn/lib:${LD_LIBRARY_PATH}
  export PYTHONPATH=/path/to/cudnn_frontend/build_python:${PYTHONPATH}
  ```

Adjust the paths above to match your local build or installation directory.

In [1]:
# !nvidia-smi

In [2]:
import torch
import torch.nn.functional as F
import cudnn

print("cuDNN backend version:", cudnn.backend_version())

cuDNN backend version: 92200


## Reference Implementation

A PyTorch reference for causal depthwise conv1d forward, used with autograd to compute reference gradients.

In [3]:
def causal_conv1d_ref(x, weight, bias, activation="silu"):
    """Reference: causal depthwise conv1d using PyTorch ops.

    Args:
        x:      (batch, dim, seq_len)
        weight: (dim, kernel_size)
        bias:   (dim,)
        activation: 'silu' or 'identity'

    Returns:
        y: (batch, dim, seq_len)
    """
    batch, dim, seq_len = x.shape
    kernel_size = weight.shape[1]

    x_padded = F.pad(x, (kernel_size - 1, 0))

    w = weight.unsqueeze(1)  # (dim, 1, kernel_size)
    y = F.conv1d(x_padded, w, bias=bias, groups=dim)

    if activation == "silu":
        y = F.silu(y)

    return y


def causal_conv1d_bwd_ref(x, weight, bias, dy, activation="silu"):
    """Reference backward via PyTorch autograd.

    Returns:
        dx, dweight, dbias (all float32 for comparison)
    """
    x = x.float().detach().requires_grad_(True)
    weight = weight.float().detach().requires_grad_(True)
    bias = bias.float().detach().requires_grad_(True)
    dy = dy.float().detach()

    y = causal_conv1d_ref(x, weight, bias, activation=activation)
    y.backward(dy)

    return x.grad, weight.grad, bias.grad

## Setup Tensors

In [4]:
batch = 2
dim = 768
seq_len = 4096
kernel_size = 4
dtype = torch.bfloat16

has_cuda = torch.cuda.is_available()

torch.manual_seed(42)

x = torch.randn(batch, dim, seq_len, dtype=dtype)
weight = torch.randn(dim, kernel_size, dtype=dtype)
bias = torch.randn(dim, dtype=dtype)
dy = torch.randn(batch, dim, seq_len, dtype=dtype)

print(f"CUDA available: {has_cuda}")
print(f"x:      {x.shape}, dtype={x.dtype}")
print(f"weight: {weight.shape}, dtype={weight.dtype}")
print(f"bias:   {bias.shape}, dtype={bias.dtype}")
print(f"dy:     {dy.shape}, dtype={dy.dtype}")

CUDA available: True
x:      torch.Size([2, 768, 4096]), dtype=torch.bfloat16
weight: torch.Size([768, 4]), dtype=torch.bfloat16
bias:   torch.Size([768]), dtype=torch.bfloat16
dy:     torch.Size([2, 768, 4096]), dtype=torch.bfloat16


## Compute Reference Gradients (CPU, via autograd)

In [5]:
dx_ref, dw_ref, db_ref = causal_conv1d_bwd_ref(x, weight, bias, dy, activation="silu")
print(f"dx_ref:  {dx_ref.shape}, dtype={dx_ref.dtype}")
print(f"dw_ref:  {dw_ref.shape}, dtype={dw_ref.dtype}")
print(f"db_ref:  {db_ref.shape}, dtype={db_ref.dtype}")

dx_ref:  torch.Size([2, 768, 4096]), dtype=torch.float32
dw_ref:  torch.Size([768, 4]), dtype=torch.float32
db_ref:  torch.Size([768]), dtype=torch.float32


## Compute cuDNN Gradients (via autograd)

In [6]:
if has_cuda:
    x_gpu = x.cuda().detach().requires_grad_(True)
    w_gpu = weight.cuda().detach().requires_grad_(True)
    b_gpu = bias.cuda().detach().requires_grad_(True)
    dy_gpu = dy.cuda()

    y_cudnn = cudnn.ops.causal_conv1d(x_gpu, w_gpu, b_gpu, activation="silu")
    y_cudnn.backward(dy_gpu)

    dx_cudnn = x_gpu.grad
    dw_cudnn = w_gpu.grad
    db_cudnn = b_gpu.grad
    print(f"dx_cudnn:  {dx_cudnn.shape}, dtype={dx_cudnn.dtype}")
    print(f"dw_cudnn:  {dw_cudnn.shape}, dtype={dw_cudnn.dtype}")
    print(f"db_cudnn:  {db_cudnn.shape}, dtype={db_cudnn.dtype}")
else:
    print("Skipping cuDNN backward (no CUDA device).")

dx_cudnn:  torch.Size([2, 768, 4096]), dtype=torch.bfloat16
dw_cudnn:  torch.Size([768, 4]), dtype=torch.bfloat16
db_cudnn:  torch.Size([768]), dtype=torch.bfloat16


## Verify Correctness

In [7]:
if has_cuda:
    atol = 1e-2
    rtol = 1e-2

    for name, cudnn_grad, ref_grad in [
        ("dx", dx_cudnn.cpu().float(), dx_ref),
        ("dweight", dw_cudnn.cpu().float(), dw_ref),
        ("dbias", db_cudnn.cpu().float(), db_ref),
    ]:
        max_abs = (cudnn_grad - ref_grad).abs().max().item()
        passed = torch.allclose(cudnn_grad, ref_grad, atol=atol, rtol=rtol)
        status = "PASS" if passed else "FAIL"
        print(f"[{status}] {name:8s} max_abs_diff={max_abs:.4e}")
        assert passed, f"{name} verification failed: max_abs={max_abs}"

    print("\nPASSED: cuDNN causal_conv1d autograd backward matches reference.")
else:
    print("Skipping verification (no CUDA device).")

[PASS] dx       max_abs_diff=3.1241e-02
[PASS] dweight  max_abs_diff=4.9881e-01
[PASS] dbias    max_abs_diff=4.9422e-01

PASSED: cuDNN causal_conv1d autograd backward matches reference.


## Test with Different Data Types and Kernel Sizes

In [8]:
tolerances = {
    torch.float32:  (5e-6, 5e-6),
    torch.float16:  (5e-3, 5e-3),
    torch.bfloat16: (1e-2, 1e-2),
}
# dweight/dbias are always FP32; error is from atomicAdd ordering
dw_db_atol, dw_db_rtol = 5e-3, 5e-3

test_configs = [
    {"dtype": torch.float32,  "kernel_size": 4,  "activation": "silu"},
    {"dtype": torch.float16,  "kernel_size": 4,  "activation": "silu"},
    {"dtype": torch.bfloat16, "kernel_size": 4,  "activation": "silu"},
    {"dtype": torch.bfloat16, "kernel_size": 3,  "activation": "silu"},
    {"dtype": torch.float16,  "kernel_size": 7,  "activation": "silu"},
    {"dtype": torch.bfloat16, "kernel_size": 4,  "activation": "identity"},
]

batch, dim, seq_len = 2, 256, 1024

for cfg in test_configs:
    dt = cfg["dtype"]
    ks = cfg["kernel_size"]
    act = cfg["activation"]
    atol, rtol = tolerances[dt]

    x = torch.randn(batch, dim, seq_len, dtype=dt)
    w = torch.randn(dim, ks, dtype=dt)
    b = torch.randn(dim, dtype=dt)
    dy = torch.randn(batch, dim, seq_len, dtype=dt)

    dx_ref, dw_ref, db_ref = causal_conv1d_bwd_ref(x, w, b, dy, activation=act)

    if has_cuda:
        x_g = x.cuda().detach().requires_grad_(True)
        w_g = w.cuda().detach().requires_grad_(True)
        b_g = b.cuda().detach().requires_grad_(True)

        y_g = cudnn.ops.causal_conv1d(x_g, w_g, b_g, activation=act)
        y_g.backward(dy.cuda())

        dx_ok = torch.allclose(x_g.grad.cpu().float(), dx_ref, atol=atol, rtol=rtol)
        dw_ok = torch.allclose(w_g.grad.cpu().float(), dw_ref, atol=max(atol, dw_db_atol), rtol=max(rtol, dw_db_rtol))
        db_ok = torch.allclose(b_g.grad.cpu().float(), db_ref, atol=max(atol, dw_db_atol), rtol=max(rtol, dw_db_rtol))
        passed = dx_ok and dw_ok and db_ok

        dx_abs = (x_g.grad.cpu().float() - dx_ref).abs().max().item()
        dw_abs = (w_g.grad.cpu().float() - dw_ref).abs().max().item()
        db_abs = (b_g.grad.cpu().float() - db_ref).abs().max().item()

        status = "PASS" if passed else "FAIL"
        print(f"[{status}] dtype={str(dt):15s} K={ks:3d} act={act:10s} "
              f"max_abs: dx={dx_abs:.4e} dw={dw_abs:.4e} db={db_abs:.4e}")
        assert passed, f"Test failed for {cfg}"
    else:
        print(f"[REF ] dtype={str(dt):15s} K={ks:3d} act={act:10s} "
              f"dx={dx_ref.shape} dw={dw_ref.shape} db={db_ref.shape}")

print("\nAll tests passed!" if has_cuda else "\nAll reference tests completed (CPU only).")

[PASS] dtype=torch.float32   K=  4 act=silu       max_abs: dx=4.2915e-06 dw=3.0518e-05 db=5.7220e-05
[PASS] dtype=torch.float16   K=  4 act=silu       max_abs: dx=3.8843e-03 dw=3.1090e-02 db=2.6917e-02
[PASS] dtype=torch.bfloat16  K=  4 act=silu       max_abs: dx=3.0970e-02 dw=2.4898e-01 db=2.3223e-01
[PASS] dtype=torch.bfloat16  K=  3 act=silu       max_abs: dx=3.1178e-02 dw=2.8888e-01 db=2.3131e-01
[PASS] dtype=torch.float16   K=  7 act=silu       max_abs: dx=5.2376e-03 dw=3.4286e-02 db=2.4826e-02


[PASS] dtype=torch.bfloat16  K=  4 act=identity   max_abs: dx=3.1250e-02 dw=4.8404e-01 db=2.4219e-01

All tests passed!


## torch.compile Support

`cudnn.ops.causal_conv1d` works with `torch.compile` for both forward and backward passes. The compiled graph captures the custom op nodes and the autograd backward.

In [9]:
if has_cuda:
    B, D, L, K = 2, 256, 1024, 4
    act = "silu"

    def train_step(x, w, b, dy):
        x = x.detach().requires_grad_(True)
        w = w.detach().requires_grad_(True)
        b = b.detach().requires_grad_(True)
        y = cudnn.ops.causal_conv1d(x, w, b, activation=act)
        y.backward(dy)
        return x.grad, w.grad, b.grad

    compiled_train_step = torch.compile(train_step)

    x_c = torch.randn(B, D, L, dtype=torch.bfloat16, device="cuda")
    w_c = torch.randn(D, K, dtype=torch.bfloat16, device="cuda")
    b_c = torch.randn(D, dtype=torch.bfloat16, device="cuda")
    dy_c = torch.randn(B, D, L, dtype=torch.bfloat16, device="cuda")

    dx_eager, dw_eager, db_eager = train_step(x_c, w_c, b_c, dy_c)
    dx_compiled, dw_compiled, db_compiled = compiled_train_step(x_c, w_c, b_c, dy_c)

    for name, eager, compiled in [
        ("dx", dx_eager, dx_compiled),
        ("dweight", dw_eager, dw_compiled),
        ("dbias", db_eager, db_compiled),
    ]:
        max_abs = (eager.float() - compiled.float()).abs().max().item()
        print(f"[{'PASS' if max_abs == 0 else 'FAIL'}] {name:8s} eager vs compiled max_abs_diff={max_abs:.4e}")
        assert max_abs == 0.0, f"{name}: torch.compile differs from eager"

    print("\nPASSED: torch.compile backward produces identical output to eager mode.")
else:
    print("Skipping torch.compile test (no CUDA device).")

[PASS] dx       eager vs compiled max_abs_diff=0.0000e+00
[PASS] dweight  eager vs compiled max_abs_diff=0.0000e+00
[PASS] dbias    eager vs compiled max_abs_diff=0.0000e+00

PASSED: torch.compile backward produces identical output to eager mode.
